# 🔍 Confluence RAG 파이프라인 단계별 디버거 & 실험실

이 노트북은 **RAG 챗봇 답변이 이상할 때 각 단계를 하나씩 눈으로 뜯어보고 디버깅**하기 위한 전용 실험 도구입니다.

---

### 📌 디버깅 4단계 흐름
1. **[1단계] Elasticsearch 연결 & 인덱스 상태 확인**
2. **[2단계] 검색 디버깅 (Retrieval)**: 내 질문으로 어떤 Confluence 문서와 청크가 검색되었는가? (유사도 점수, 제목, 경로, 본문 발췌)
3. **[3단계] 프롬프트 디버깅 (Prompt Context)**: 검색된 청크들이 LLM 프롬프트에 어떻게 합쳐졌는가?
4. **[4단계] 답변 생성 디버깅 (Generation)**: LLM(DeepSeek / GPT)이 참고 문서를 바탕으로 어떤 답변을 내놓았는가?
5. **[5단계] 멀티턴 질문 재작성 디버깅 (Query Rewrite)**: 앞선 대화 맥락을 보고 질문을 제대로 재작성했는가?

In [ ]:
# [1단계] 경로 설정 및 필수 모듈 임포트
import sys
import os
from pathlib import Path
import pandas as pd

# ai-server 경로 추가
ai_server_path = str(Path(os.getcwd()).parent / "ai-server")
if ai_server_path not in sys.path:
    sys.path.append(ai_server_path)

from app.config import settings
from app.retrieval.es_client import get_es_client, search_hybrid
from app.llm.litellm_client import embed_texts, generate_answer, generate_chat_completion

# Elasticsearch 연결 확인
es = get_es_client()
is_connected = es.ping()
doc_count = es.count(index=settings.ELASTICSEARCH_INDEX)["count"] if is_connected else 0

print(f"✅ [Elasticsearch 연결]: {'성공' if is_connected else '실패'}")
print(f"📚 [현재 색인된 인덱스]: {settings.ELASTICSEARCH_INDEX}")
print(f"📄 [총 색인된 청크 수]: {doc_count:,}개")

--- 
## 2. 검색(Retrieval) 디버깅: 원하는 질문을 넣고 검색 결과를 직접 확인해보세요!

In [ ]:
# 🔍 테스트하고 싶은 질문을 아래에 입력하세요
TEST_QUERY = "연차 및 반차 신청 규정과 방법 알려줘"
TOP_K = 3  # 상위 몇 개 문서를 가져올지

print(f"[질문]: {TEST_QUERY}")
print(f"[임베딩 생성 중... (OpenAI text-embedding-3-small)]")
query_vector = embed_texts([TEST_QUERY])[0]

# 하이브리드 검색 실행 (BM25 + Vector kNN)
search_hits = search_hybrid(TEST_QUERY, query_vector=query_vector, top_k=TOP_K)

print(f"\n🎯 검색된 문서 수: {len(search_hits)}개\n")

# 검색 결과 요약 테이블 출력
summary_data = []
for i, hit in enumerate(search_hits, 1):
    summary_data.append({
        "순위": f"{i}위",
        "유사도 점수": f"{hit['score']:.2f}",
        "문서 제목": hit["title"],
        "카테고리": hit.get("category") or "-",
        "계층 경로": hit.get("path") or "-",
        "작성자": hit.get("author") or "-"
    })

display(pd.DataFrame(summary_data))

# 각 청크의 본문 내용 상세 출력
print("=" * 70)
for i, hit in enumerate(search_hits, 1):
    print(f"\n📄 [{i}위 문서 청크 상세] - {hit['title']} (Score: {hit['score']:.2f})")
    print(f"🔗 URL: {hit.get('url')}")
    print(f"📁 경로: {hit.get('path')}")
    print("-" * 70)
    print(hit["text"])  # 청크 본문 (표 마크다운 포함)
    print("=" * 70)

--- 
## 3. 프롬프트 조립 & 최종 LLM 답변 생성 디버깅
검색된 문서들을 바탕으로 프롬프트가 어떻게 합쳐지고, LLM이 어떤 답변을 생성하는지 확인합니다.

In [ ]:
# 1. 검색된 청크들을 결합하여 Context 생성
context_text = "\n\n---\n\n".join([
    f"[문서 제목: {hit['title']}] (카테고리: {hit.get('category', '')}, 경로: {hit.get('path', '')})\n{hit['text']}"
    for hit in search_hits
])

print("🚀 [LLM 답변 생성 요청 중...] (LiteLLM Gateway -> generate_answer)")
llm_answer = generate_answer(query=TEST_QUERY, context=context_text)

print("\n" + "=" * 70)
print("🤖 [AI 챗봇 최종 생성 답변]")
print("=" * 70)
print(llm_answer)
print("=" * 70)

--- 
## 4. 멀티턴 질문 재작성(Query Rewrite) 디버깅
직전 대화가 있을 때, 대명사("그 사람", "그거")가 포함된 후속 질문이 올바른 단독 질문으로 재작성되는지 테스트합니다.

In [ ]:
# 가상의 직전 대화 기록 (History)
mock_history = [
    {"role": "user", "content": "회사 연혁 알려줘"},
    {"role": "assistant", "content": "2026년 1월 청년일자리 강소기업에 선정되었고, 2025년 9월 벤처기업인증을 재선정받았습니다."}
]

# 사용자의 후속 질문 (대명사 포함)
follow_up_query = "거기서 2024년 표창 수상 내역만 자세히 알려줘"

# 질문 재작성 프롬프트
rewrite_system_prompt = """당신은 대화 맥락을 파악하여 사용자의 후속 질문을 독립적인 검색용 질문(Stand-alone Query)으로 재작성하는 전문가입니다.
대화 히스토리를 참고하여 '거기서', '그 사람', '그거' 등의 지시대명사를 구체적인 대상명으로 바꾸어 하나의 완성된 검색 문장으로만 출력하세요.
다른 설명이나 인사말은 일절 붙이지 마세요."""

history_str = "\n".join([f"{h['role']}: {h['content']}" for h in mock_history])
rewrite_user_prompt = f"[이전 대화]\n{history_str}\n\n[사용자의 새 질문]\n{follow_up_query}\n\n[재작성된 독립 질문]:"

rewrite_messages = [
    {"role": "system", "content": rewrite_system_prompt},
    {"role": "user", "content": rewrite_user_prompt}
]

rewritten = generate_chat_completion(rewrite_messages).strip()

print(f"🗣️ [원본 후속 질문]: {follow_up_query}")
print(f"✨ [재작성된 검색 질문]: {rewritten}")

# 재작성된 질문으로 실제 검색도 즉시 실행해보기
rewritten_vector = embed_texts([rewritten])[0]
rewritten_hits = search_hybrid(rewritten, query_vector=rewritten_vector, top_k=2)
print(f"\n🔎 [재작성 질문으로 검색된 문서 Top 2]:")
for h in rewritten_hits:
    print(f" - {h['title']} (Score: {h['score']:.2f})")

--- 
## 5. 🛠️ 원클릭 RAG 디버거 함수 (자주 쓰는 질문 테스트용)
질문 문자열만 넣으면 검색된 문서와 최종 답변을 한 번에 예쁘게 출력해 주는 종합 헬퍼 함수입니다.

In [ ]:
def test_rag(query: str, top_k: int = 3):
    """
    원하는 질문에 대해 검색된 문서 목록과 생성된 답변을 한 번에 확인하는 함수
    """
    print(f"\n" + "="*80)
    print(f"❓ [질문]: {query}")
    print("="*80)
    
    # 1. 임베딩 및 검색
    v = embed_texts([query])[0]
    hits = search_hybrid(query, query_vector=v, top_k=top_k)
    
    print(f"\n📚 [검색된 참고 문서 Top {len(hits)}]")
    for i, h in enumerate(hits, 1):
        print(f"  {i}. [{h['title']}] (점수: {h['score']:.2f}) | 카테고리: {h.get('category')} | 경로: {h.get('path')}")
    
    # 2. 답변 생성
    ctx = "\n\n".join([f"[제목: {h['title']}]\n{h['text']}" for h in hits])
    ans = generate_answer(query=query, context=ctx)
    
    print("\n🤖 [생성된 답변]")
    print("-"*80)
    print(ans)
    print("="*80)

# 테스트 실행 예시 (원하시는 질문으로 자유롭게 바꿔보세요!)
test_rag("회사 연혁에 대해 알려줘")